# Đề tài: Tiền xử lý văn bản tiếng Việt cho bài toán gom cụm văn bản báo chí

## Bối cảnh
Sau Notebook 1, bộ dữ liệu đã được gộp, loại trùng và lọc bài quá ngắn để tạo thành file chính sạch `bai_test_do_an_main_clean.csv`. Tuy nhiên, để phục vụ tốt cho bài toán gom cụm văn bản tiếng Việt, dữ liệu vẫn cần được tiền xử lý sâu hơn ở mức ngôn ngữ, đặc biệt là trong bối cảnh tiếng Việt có đặc trưng đa âm tiết, nhiều từ ghép và mức độ chồng lấn chủ đề giữa các chuyên mục.

## Mục tiêu
Notebook này tập trung vào việc xây dựng tập văn bản đầu vào chuẩn cho giai đoạn gom cụm, bao gồm:
- tạo cột văn bản chính từ `title`, `description` và `content`,
- chuẩn hóa văn bản tiếng Việt,
- tách từ tiếng Việt (word segmentation),
- loại bỏ từ dừng ở nhánh lexical,
- tạo các cột văn bản phù hợp cho các hướng biểu diễn đặc trưng như TF-IDF và BM25.

## Ý nghĩa học thuật
Tiền xử lý là bước có ảnh hưởng trực tiếp đến chất lượng biểu diễn văn bản và độ tách biệt của cụm. Với dữ liệu báo chí tiếng Việt, việc xây dựng một pipeline tiền xử lý nhất quán giúp giảm nhiễu, bảo toàn tín hiệu chủ đề và tạo nền tảng đáng tin cậy cho các bước gom cụm, đánh giá và phân tích cụm ở notebook tiếp theo.

## Đầu ra mong đợi
Sau notebook này, đầu ra mong đợi gồm:
- một file dữ liệu đã được bổ sung các cột văn bản phục vụ NLP,
- một cột văn bản gốc ghép từ nhiều trường nội dung,
- một cột văn bản đã chuẩn hóa,
- một cột văn bản đã tách từ tiếng Việt,
- và một cột văn bản đã làm sạch để sẵn sàng cho nhánh lexical trong bước gom cụm.

In [1]:
# Cell 2: import thư viện và khai báo đường dẫn dữ liệu cho Notebook 2
# Cell này dùng để chuẩn bị môi trường làm việc và xác định file sạch đầu vào cùng nơi lưu file kết quả sau tiền xử lý.

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

#INPUT_PATH = Path("D:/BAI TEST/data/processed_data/bai_test_do_an_main_clean.csv")
#OUTPUT_DIR = Path("D:/BAI TEST/data/processed_data")
INPUT_PATH = Path("bai_test_do_an_main_clean.csv")

OUTPUT_DIR = Path("processed_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREPROCESSED = OUTPUT_DIR / "bai_test_do_an_preprocessed.csv"

print("File đầu vào:", INPUT_PATH)
print("Tồn tại:", INPUT_PATH.exists())

print("\nThư mục đầu ra:", OUTPUT_DIR)
print("File sẽ lưu tại:", OUTPUT_PREPROCESSED)

File đầu vào: bai_test_do_an_main_clean.csv
Tồn tại: True

Thư mục đầu ra: processed_data
File sẽ lưu tại: processed_data/bai_test_do_an_preprocessed.csv


In [2]:
# Cell 3: đọc file dữ liệu sạch và kiểm tra nhanh cấu trúc đầu vào
# Cell này giúp xác nhận file đầu vào đã đúng và sẵn sàng cho bước tạo văn bản chính phục vụ NLP.

df = pd.read_csv(INPUT_PATH)

print("Kích thước dữ liệu:", df.shape)
print("\nCác cột hiện có:")
print(df.columns.tolist())

print("\nSố giá trị thiếu theo cột:")
display(df.isna().sum().to_frame("missing_count"))

display(df.head(3))

Kích thước dữ liệu: (8727, 8)

Các cột hiện có:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url']

Số giá trị thiếu theo cột:


,missing_count
doc_id,0
source,0
category_clean,0
title,0
description,0
content,0
published_date,0
url,0


,doc_id,source,category_clean,title,description,content,published_date,url
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...


In [3]:
# Cell 4: tạo cột văn bản chính cho bài toán NLP
# Cell này ghép title, description và content thành một trường văn bản thống nhất để dùng cho các bước chuẩn hóa và tách từ phía sau.

df_work = df.copy()

df_work["cluster_text"] = (
    df_work["title"].astype(str).str.strip() + " . " +
    df_work["description"].astype(str).str.strip() + " . " +
    df_work["content"].astype(str).str.strip()
)

print("Kích thước dữ liệu làm việc:", df_work.shape)
print("\nCác cột hiện có sau khi tạo cluster_text:")
print(df_work.columns.tolist())

display(
    df_work[["doc_id", "category_clean", "title", "description", "content", "cluster_text"]].head(2)
)

Kích thước dữ liệu làm việc: (8727, 9)

Các cột hiện có sau khi tạo cluster_text:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'cluster_text']


,doc_id,category_clean,title,description,content,cluster_text
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,Meta đưa AI vào Messenger trả lời khách hàng 2...


In [4]:
# Cell 5: định nghĩa hàm chuẩn hóa văn bản mức nhẹ
# Cell này tạo bước làm sạch cơ sở để giảm nhiễu nhưng vẫn giữ lại phần lớn tín hiệu ngôn ngữ cần cho gom cụm.

def normalize_vietnamese_text(text):
    text = str(text)

    # Chuẩn hóa Unicode
    text = unicodedata.normalize("NFC", text)

    # Bỏ HTML nếu có
    text = re.sub(r"<[^>]+>", " ", text)

    # Bỏ URL nếu có
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)

    # Chuẩn hóa xuống chữ thường
    text = text.lower()

    # Thay các ký tự xuống dòng, tab bằng khoảng trắng
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Giữ chữ, số, khoảng trắng và một số dấu câu cơ bản để tránh làm dính từ quá sớm
    text = re.sub(r"[^0-9a-zA-ZÀ-Ỹà-ỹ\s\.\,\!\?\:\;\-\%\/]", " ", text)

    # Chuẩn hóa nhiều khoảng trắng liên tiếp
    text = re.sub(r"\s+", " ", text).strip()

    return text

print("Đã sẵn sàng hàm normalize_vietnamese_text()")

Đã sẵn sàng hàm normalize_vietnamese_text()


In [5]:
# Cell 6: áp dụng chuẩn hóa văn bản cho cột cluster_text
# Cell này tạo ra cột cluster_text_clean để dùng làm nền cho bước tách từ tiếng Việt ở cell sau.

df_work["cluster_text_clean"] = df_work["cluster_text"].apply(normalize_vietnamese_text)

print("Kích thước dữ liệu sau khi tạo cluster_text_clean:", df_work.shape)

display(
    df_work[["doc_id", "category_clean", "cluster_text", "cluster_text_clean"]].head(2)
)

Kích thước dữ liệu sau khi tạo cluster_text_clean: (8727, 10)


,doc_id,category_clean,cluster_text,cluster_text_clean
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,làn sóng máy tính lượng tử sau cơn sốt ai . qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả lời khách hàng 2...


In [7]:
# Cell 7: cài đặt/import PyVi và tách từ tiếng Việt
# Cell này thực hiện bước quan trọng của tiền xử lý tiếng Việt: nhận diện từ ghép và tạo cột cluster_text_segmented
# để chuẩn bị cho các nhánh biểu diễn lexical như TF-IDF và BM25.

try:
    from pyvi import ViTokenizer
    print("Đã import pyvi thành công.")
except ImportError:
    !pip -q install pyvi
    from pyvi import ViTokenizer
    print("Đã cài đặt và import pyvi thành công.")

import time

def segment_vietnamese_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    return ViTokenizer.tokenize(text)

print("\nĐang tiến hành tách từ cho toàn bộ văn bản... (quá trình này có thể mất 1-2 phút)")
start_time = time.time()

df_work["cluster_text_segmented"] = df_work["cluster_text_clean"].apply(segment_vietnamese_text)

elapsed_time = time.time() - start_time

print(f"Hoàn thành tách từ trong: {elapsed_time:.2f} giây")
print("Kích thước dữ liệu hiện tại:", df_work.shape)

print("\nSo sánh văn bản trước và sau khi tách từ (dòng 0):")
print("- Gốc sạch :", df_work.loc[0, "cluster_text_clean"][:200], "...")
print("- Tách từ  :", df_work.loc[0, "cluster_text_segmented"][:200], "...")

display(
    df_work[["doc_id", "category_clean", "cluster_text_segmented"]].head(2)
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 83.9 MB/s eta 0:00:00
Đã cài đặt và import pyvi thành công.

Đang tiến hành tách từ cho toàn bộ văn bản... (quá trình này có thể mất 1-2 phút)
Hoàn thành tách từ trong: 93.85 giây
Kích thước dữ liệu hiện tại: (8727, 11)

So sánh văn bản trước và sau khi tách từ (dòng 0):
- Gốc sạch : làn sóng máy tính lượng tử sau cơn sốt ai . quý 1-2026, thế giới chứng kiến loạt công ty vi tính lượng tử liên tiếp niêm yết trên các sàn chứng khoán mỹ, được xem là tín hiệu rõ rệt cho thấy công nghệ ...
- Tách từ  : làn_sóng máy_tính lượng_tử sau cơn_sốt ai . quý 1 - 2026 , thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết trên các sàn chứng_khoán mỹ , được xem là tín_hiệu rõ_rệt cho thấy công_ ...


,doc_id,category_clean,cluster_text_segmented
0,1,cong_nghe,làn_sóng máy_tính lượng_tử sau cơn_sốt ai . qu...
1,2,cong_nghe,meta đưa ai vào messenger trả_lời khách_hàng 2...


### Nhận xét nhanh sau bước tách từ tiếng Việt

Bước tách từ bằng `PyVi` đã hoạt động đúng như kỳ vọng. Các cụm từ đa âm tiết quan trọng như `làn_sóng`, `máy_tính`, `lượng_tử`, `chứng_khoán`, `tín_hiệu` đã được giữ lại dưới dạng token ghép, giúp bảo toàn ý nghĩa ngữ nghĩa tốt hơn so với việc tách rời từng âm tiết.

Đây là tín hiệu tích cực cho các bước biểu diễn văn bản theo hướng lexical như **TF-IDF** và **BM25**, vì các mô hình này rất nhạy với chất lượng token đầu vào.

In [8]:
# Cell 8: tạo danh sách ứng viên stopwords từ chính corpus đã tách từ
# Cell này dùng document frequency và độ phủ theo chuyên mục để tìm các token quá phổ biến,
# từ đó hỗ trợ xây dựng bộ stopwords phù hợp riêng cho dữ liệu báo chí của đồ án.

from collections import Counter, defaultdict

token_doc_freq = Counter()
token_category_set = defaultdict(set)

for _, row in df_work[["category_clean", "cluster_text_segmented"]].iterrows():
    category = row["category_clean"]
    text = row["cluster_text_segmented"]

    if not isinstance(text, str) or text.strip() == "":
        continue

    tokens = set(text.split())

    for token in tokens:
        token_doc_freq[token] += 1
        token_category_set[token].add(category)

candidate_rows = []
n_docs = len(df_work)
n_categories = df_work["category_clean"].nunique()

for token, dfreq in token_doc_freq.items():
    doc_ratio = dfreq / n_docs
    category_coverage = len(token_category_set[token]) / n_categories

    candidate_rows.append({
        "token": token,
        "doc_freq": dfreq,
        "doc_ratio": round(doc_ratio, 4),
        "category_count": len(token_category_set[token]),
        "category_coverage": round(category_coverage, 4),
        "token_length": len(token)
    })

candidate_stopwords = pd.DataFrame(candidate_rows)

# lọc các token quá phổ biến và phủ rộng nhiều chuyên mục
candidate_stopwords = candidate_stopwords[
    (candidate_stopwords["doc_ratio"] >= 0.20) &
    (candidate_stopwords["category_coverage"] >= 0.80)
].copy()

# loại bớt token quá ngắn hoặc chỉ là dấu câu
candidate_stopwords = candidate_stopwords[
    candidate_stopwords["token"].str.contains(r"[a-zA-ZÀ-Ỹà-ỹ0-9_]", regex=True)
]

candidate_stopwords = candidate_stopwords.sort_values(
    by=["doc_ratio", "category_coverage", "doc_freq"],
    ascending=False
).reset_index(drop=True)

print("Số lượng ứng viên stopwords:", len(candidate_stopwords))
display(candidate_stopwords.head(100))

Số lượng ứng viên stopwords: 201


,token,doc_freq,doc_ratio,category_count,category_coverage,token_length
0,và,8656,0.9919,6,1.0,2
1,trong,8349,0.9567,6,1.0,5
2,của,8250,0.9453,6,1.0,3
3,các,8167,0.9358,6,1.0,3
4,cho,8155,0.9345,6,1.0,3
...,...,...,...,...,...,...
95,2026,2820,0.3231,6,1.0,4
96,nếu,2807,0.3216,6,1.0,3
97,đồng_thời,2804,0.3213,6,1.0,9
98,giữa,2795,0.3203,6,1.0,4


In [9]:
# Cell 9: xem kỹ nhóm ứng viên stopwords phổ biến nhất để chốt danh sách tùy chỉnh
# Cell này giúp quan sát trực tiếp các token phủ rộng toàn corpus trước khi quyết định loại bỏ khỏi nhánh lexical.

top_n = 120

candidate_preview = candidate_stopwords.head(top_n).copy()

print(f"Hiển thị {top_n} ứng viên stopwords đầu tiên:")
display(candidate_preview)

print("\nDanh sách token top đầu:")
print(candidate_preview["token"].tolist())

Hiển thị 120 ứng viên stopwords đầu tiên:


,token,doc_freq,doc_ratio,category_count,category_coverage,token_length
0,và,8656,0.9919,6,1.0,2
1,trong,8349,0.9567,6,1.0,5
2,của,8250,0.9453,6,1.0,3
3,các,8167,0.9358,6,1.0,3
4,cho,8155,0.9345,6,1.0,3
...,...,...,...,...,...,...
115,5,2570,0.2945,6,1.0,1
116,thông_tin,2561,0.2935,6,1.0,9
117,tiếp_tục,2561,0.2935,6,1.0,8
118,khiến,2531,0.2900,6,1.0,5



Danh sách token top đầu:
['và', 'trong', 'của', 'các', 'cho', 'được', 'với', 'là', 'có', 'đến', 'đã', 'để', 'từ', 'đó', 'không', 'khi', 'này', 'theo', 'tại', 'nhiều', 'một', 'về', 'trên', 'người', 'ở', 'vào', 'cũng', 'những', 'ngày', 'như', 'ra', 'hơn', 'còn', 'sau', 'năm', 'việc', 'sẽ', 'chỉ', 'trước', 'đây', 'biết', 'lại', 'đang', 'sự', 'phải', 'cao', 'mới', 'nhưng', 'nhất', 'có_thể', 'cùng', 'mà', 'qua', 'làm', '3', 'do', '2', 'lớn', '1', 'vẫn', 'lên', 'đi', 'hai', 'bị', 'cần', 'đặc_biệt', 'rất', 'tổ_chức', 'khác', 'ông', 'hay', 'cả', 'tạo', 'điểm', 'thời_gian', 'giúp', 'gần', 'hoặc', 'tới', 'thấy', 'tp', 'nam', 'nhà', 'số', 'chưa', 'đưa', 'từng', 'việt_nam', 'khoảng', 'nên', 'nước', 'phát_triển', 'lần', 'mang', 'hoạt_động', '2026', 'nếu', 'đồng_thời', 'giữa', 'bên', '2025', '4', 'bằng', 'thêm', 'nói', 'rằng', 'bộ', 'tuy_nhiên', 'đầu', 'tháng', '10', 'ngay', 'mỗi', 'nguyễn', 'sử_dụng', '5', 'thông_tin', 'tiếp_tục', 'khiến', 'vừa']


In [10]:
# Cell 10: chốt bộ stopwords tùy chỉnh từ chính corpus và tạo cột lexical
# Cell này tạo ra cluster_text_lexical cho nhánh TF-IDF/BM25 bằng cách loại bỏ
# các token chức năng và stopwords kiểu báo chí quá phổ biến trong dữ liệu.

custom_stopwords = {
    "và", "trong", "của", "các", "cho", "được", "với", "là", "có", "đến",
    "đã", "để", "từ", "đó", "không", "khi", "này", "theo", "tại", "nhiều",
    "một", "về", "trên", "ở", "vào", "cũng", "những", "như", "ra", "hơn",
    "còn", "sau", "năm", "việc", "sẽ", "chỉ", "trước", "đây", "lại", "đang",
    "sự", "phải", "nhưng", "mà", "qua", "do", "vẫn", "lên", "đi", "hai",
    "bị", "cần", "rất", "khác", "hay", "cả", "gần", "hoặc", "tới", "chưa",
    "từng", "khoảng", "nên", "lần", "nếu", "giữa", "bên", "bằng", "thêm",
    "rằng", "tuy_nhiên", "đầu", "tháng", "ngay", "mỗi", "vừa",

    # stopwords kiểu báo chí trong corpus
    "ngày", "thời_gian", "thông_tin", "tiếp_tục", "thấy", "biết",

    # token số quá phổ biến
    "1", "2", "3", "4", "5", "10", "2025", "2026"
}

punctuation_tokens = {
    ".", ",", "!", "?", ":", ";", "-", "--", "%", "/", "(", ")", "[", "]", "{", "}"
}

stopword_set = custom_stopwords.union(punctuation_tokens)

def remove_stopwords(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    tokens = text.split()
    filtered_tokens = []

    for token in tokens:
        token = token.strip()
        if not token:
            continue
        if token in stopword_set:
            continue
        if len(token) == 1 and not token.isalnum():
            continue
        filtered_tokens.append(token)

    return " ".join(filtered_tokens)

df_work["cluster_text_lexical"] = df_work["cluster_text_segmented"].apply(remove_stopwords)

print("Đã tạo xong cột cluster_text_lexical")
print("Kích thước dữ liệu hiện tại:", df_work.shape)

print("\nSo sánh dòng 0 trước và sau khi xóa stopwords:")
print("- Segmented :", df_work.loc[0, "cluster_text_segmented"][:220], "...")
print("- Lexical   :", df_work.loc[0, "cluster_text_lexical"][:220], "...")

display(
    df_work[["doc_id", "category_clean", "cluster_text_lexical"]].head(2)
)

Đã tạo xong cột cluster_text_lexical
Kích thước dữ liệu hiện tại: (8727, 12)

So sánh dòng 0 trước và sau khi xóa stopwords:
- Segmented : làn_sóng máy_tính lượng_tử sau cơn_sốt ai . quý 1 - 2026 , thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết trên các sàn chứng_khoán mỹ , được xem là tín_hiệu rõ_rệt cho thấy công_nghệ này đang có nhữ ...
- Lexical   : làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết sàn chứng_khoán mỹ xem tín_hiệu rõ_rệt công_nghệ bước_đi đầu_tiên hành_trình thương_mại_hóa đáng chú_ý giai_đ ...


,doc_id,category_clean,cluster_text_lexical
0,1,cong_nghe,làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_...
1,2,cong_nghe,meta đưa ai messenger trả_lời khách_hàng 24 7 ...


In [12]:
# Cell 11: chọn các cột cần thiết và lưu file tiền xử lý cuối cùng của Notebook 2
# Cell này chốt dữ liệu NLP đầu ra để dùng ổn định cho các notebook gom cụm tiếp theo.

final_columns = [
    "doc_id",
    "source",
    "category_clean",
    "published_date",
    "url",
    "cluster_text",          # Nguyên bản (để hiển thị, đối chiếu)
    "cluster_text_clean",     # Chuẩn hóa nhẹ (tùy chọn)
    "cluster_text_segmented", # Đã tách từ nhưng còn stopwords (Dùng cho PhoBERT nếu cần)
    "cluster_text_lexical"    # Tách từ + Đã xóa stopwords (Vũ khí chính cho TF-IDF/BM25)
]

df_final_nlp = df_work[final_columns].copy()

df_final_nlp.to_csv(OUTPUT_PREPROCESSED, index=False, encoding="utf-8-sig")

print("Kích thước file NLP cuối cùng:", df_final_nlp.shape)
print("\nĐã lưu file tại:")
print(OUTPUT_PREPROCESSED)

print("\nCác cột đã lưu:")
print(df_final_nlp.columns.tolist())

display(df_final_nlp.head(3))

Kích thước file NLP cuối cùng: (8727, 9)

Đã lưu file tại:
processed_data/bai_test_do_an_preprocessed.csv

Các cột đã lưu:
['doc_id', 'source', 'category_clean', 'published_date', 'url', 'cluster_text', 'cluster_text_clean', 'cluster_text_segmented', 'cluster_text_lexical']


,doc_id,source,category_clean,published_date,url,cluster_text,cluster_text_clean,cluster_text_segmented,cluster_text_lexical
0,1,tuoitre,cong_nghe,2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,làn sóng máy tính lượng tử sau cơn sốt ai . qu...,làn_sóng máy_tính lượng_tử sau cơn_sốt ai . qu...,làn_sóng máy_tính lượng_tử cơn_sốt ai quý thế_...
1,2,tuoitre,cong_nghe,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,Meta đưa AI vào Messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả_lời khách_hàng 2...,meta đưa ai messenger trả_lời khách_hàng 24 7 ...
2,3,tuoitre,cong_nghe,2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,hà tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...
